# Issue 003 — Deterministic crop clip generator

End-to-end notebook for the Issue 002 → Issue 003 seam:

1. Mount Drive + project layout.
2. Locate the Issue 002 tracking artefacts and the URFD debug manifest.
3. Build deterministic WebDataset-style `.tar` shards of fixed-length
   person crop clips from cam0 tracks (Issue 002 close-out decision).
4. No re-detection — Issue 002 boxes are consumed verbatim.
5. Write per-clip and run-level summaries to Drive.

Output goes to `MyDrive/fall_detection/artifacts/crops/` so it sits
next to the Issue 002 tracking artefacts (per Issue 001's Drive layout).

Issue 003 hard rules:
  - Same input + same config → same shard bytes.
  - Configurable clip length (16 or 32), default 32.
  - Configurable margin in [0.20, 0.40], default 0.30.
  - Configurable target size, default 224×224.
  - Skip windows with insufficient coverage or excessive gaps; log
    the reason in the run summary.
  - All clip metadata needed by Issue 006/009/011 lives in the
    `.meta.json` sidecar inside each shard.

## 1. Mount Drive + select data mode

We use **LOCAL mode** by default: datasets live on the Colab
local disk so the active pipeline avoids Drive FUSE small-file
reads. Final artefacts still land on Drive so they survive a
runtime reset.

Override the mode via the ``FALL_DETECTION_DATA_MODE`` env var
(``local`` / ``drive``) or by passing a different root to
``resolve_data_layout``.


In [ ]:
import os, pathlib, sys

# Mount Drive for artefact persistence (always needed).
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not on Colab — Drive mount skipped.")

PROJECT_ROOT = pathlib.Path(os.environ.get("FALL_DETECTION_PROJECT_ROOT", "/content/fall_detection"))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from colab.data_mode import (
    DataMode,
    resolve_data_layout,
    describe_layout,
)

DATA_MODE = DataMode(os.environ.get("FALL_DETECTION_DATA_MODE", "local").lower())
layout = resolve_data_layout(mode=DATA_MODE)
layout.ensure()

print(f"Data mode        : {layout.mode.value}")
print(f"Dataset root     : {layout.dataset_root}  (active reads happen here)")
print(f"Artefact root    : {layout.artifact_root}  (writes land here)")
print(f"Layout summary   : {describe_layout(layout)}")
if DATA_MODE is DataMode.LOCAL:
    print("Active processing reads from LOCAL disk; only artefacts persist on Drive.")


## 2. Resolve Issue 002 artefacts + manifest paths

We assume Issue 002 has been run on Drive. If not, run
`colab/002_perception_urfd.ipynb` first.

In [ ]:
from pathlib import Path
from data.manifests import load_manifest

PERCEPTION_ROOT = layout.artifacts / "perception"
CROPS_ROOT = layout.artifacts / "crops"
CROPS_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = layout.datasets / "urfd" / "manifest.yaml"

manifest = load_manifest(MANIFEST_PATH)
print(f"Manifest: {MANIFEST_PATH}")
print(f"  clips          : {len(manifest.clips)}")
print(f"  perception root: {PERCEPTION_ROOT}")
print(f"  crops root     : {CROPS_ROOT}")

# Quick sanity: which clips have perception artefacts on Drive?
ready = []
missing = []
for clip in manifest.clips:
    detection_path = PERCEPTION_ROOT / clip.clip_id / f"{clip.clip_id}_detections.json"
    if detection_path.is_file():
        ready.append(clip.clip_id)
    else:
        missing.append(clip.clip_id)
print(f"\nClips with detections: {len(ready)}")
print(f"Clips missing detections: {len(missing)}")

## 3. Configure crop parameters

Tweak these if you want a different shape / margin / length. Defaults
match the PRD's recommended fixed clip contract.

In [ ]:
from cropping.clip_builder import CropConfig

crop_config = CropConfig(
    output_size=224,   # 224x224 (PRD default)
    margin=0.30,       # mid-range of PRD [0.20, 0.40]
    clip_length=32,    # 32 frames (PRD: 16 or 32)
)
print(f"Crop config: {crop_config}")

## 4. Run the cropping pipeline

Filter to URFD cam0 tracks per the Issue 002 close-out decision
(cam0 is the downstream path; cam1 stays as a detection-limited hard slice).
The runner is fully deterministic: same input + same config produces
the same shards.

In [ ]:
from cropping.runner import run_cropping, write_run_summary

summary = run_cropping(
    layout_root=layout.root,
    perception_root=PERCEPTION_ROOT,
    crops_root=CROPS_ROOT,
    manifest_path=MANIFEST_PATH,
    crop_config=crop_config,
    camera_filter="cam0",  # Issue 002 close-out decision
    max_shards=9999,
)

summary_path = CROPS_ROOT / "_run_summary.json"
write_run_summary(summary, summary_path)
print(f"Run summary: {summary_path}")
print(f"  clips processed : {summary.clips_processed}")
print(f"  tracks processed: {summary.tracks_processed}")
print(f"  windows emitted : {summary.windows_emitted}")
print(f"  windows skipped : {summary.windows_skipped}")
print(f"  skip reasons    : {summary.skip_reason_counts}")
print(f"  shards written  : {summary.shards_written}")
print(f"  elapsed seconds : {summary.elapsed_seconds:.1f}")

## 5. Verify a shard round-trips

Sanity check: read one shard back, confirm the image + metadata members
are present and the metadata contract is complete.

In [ ]:
from cropping.shard_writer import read_shard

shards = sorted(CROPS_ROOT.glob("shard-*.tar"))
if not shards:
    print("No shards written — nothing to verify.")
else:
    sample = shards[0]
    result = read_shard(sample)
    print(f"Sampled: {sample.name}")
    print(f"  image members : {len(result.image_members)}")
    print(f"  meta members  : {len(result.metadata_members)}")
    print(f"  manifest keys : {sorted(result.manifest.keys())}")
    sample_meta_member = next(iter(result.metadata_members))
    sample_meta = result.metadata_members[sample_meta_member]
    print(f"\nSample metadata ({sample_meta_member}):")
    for key in ("clip_id", "dataset", "label", "track_id",
                "frame_index", "coverage", "crop_config",
                "margin_used", "shard_filename"):
        print(f"  {key:<15}: {sample_meta.get(key)}")

## 6. Done

Issue 003 deliverables on Drive:

  - `MyDrive/fall_detection/artifacts/crops/shard-NNNNNN.tar` —
    WebDataset-style shards containing crop JPEG frames + JSON
    metadata sidecars (one pair per frame, plus a `_manifest.json`).
  - `MyDrive/fall_detection/artifacts/crops/_run_summary.json` —
    per-clip and run-level counts (clips processed, tracks
    processed, windows emitted, windows skipped with reasons,
    shards written, runtime).

Downstream readiness:
  - Issue 005 (Pipeline A data prep) can read these shards directly.
  - Issue 008 (Pipeline B data prep / skeleton extraction) shares the
    same source frame folder + manifest; it doesn't need Issue 003
    shards because it works on raw crops, not the per-track pipeline.